# Module 16 Lab: Prompt Engineering & Advanced Prompting Patterns

**Mục tiêu lab:** Xây dựng complete prompt engineering pipeline cho LoanBot:
- Structured 5-layer system prompt
- Chain-of-Thought reasoning simulation
- Few-shot exemplar design và evaluation
- Structured output parsing và validation
- Self-critique loop và prompt A/B testing

**Nhân vật:** FinTech Corp / LoanBot v3.2 — tập trung vào prompt quality

**Hướng dẫn:** Chạy từng cell theo thứ tự. Section 6 là bài tập tự thực hành.

## Section 1: Setup & Prompt Architecture

In [ ]:
import json
import re
import random
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from enum import Enum

# 4 standard test cases
LOAN_CASES = {
    'FC-2024-001': {
        'amount_vnd': 500_000_000,
        'credit_score': 720, 'credit_grade': 'A',
        'monthly_income': 45_000_000, 'income_verified': True,
        'compliance': {'aml_clear': True, 'blacklisted': False, 'risk_level': 'LOW'},
        'property_ltv': 0.059, 'dti': 0.082,
        'expected_decision': 'APPROVE'
    },
    'FC-2024-002': {
        'amount_vnd': 200_000_000,
        'credit_score': 580, 'credit_grade': 'C',
        'monthly_income': 18_000_000, 'income_verified': True,
        'compliance': {'aml_clear': True, 'blacklisted': False, 'risk_level': 'MEDIUM'},
        'property_ltv': None, 'dti': 0.52,
        'expected_decision': 'REVIEW'
    },
    'FC-2024-003': {
        'amount_vnd': 100_000_000,
        'credit_score': 400, 'credit_grade': 'D',
        'monthly_income': 8_000_000, 'income_verified': True,
        'compliance': {'aml_clear': False, 'blacklisted': True, 'risk_level': 'HIGH'},
        'property_ltv': None, 'dti': 0.61,
        'expected_decision': 'REJECT'
    },
    'FC-2024-004': {
        'amount_vnd': 350_000_000,
        'credit_score': 645, 'credit_grade': 'B',
        'monthly_income': 31_000_000, 'income_verified': True,
        'compliance': {'aml_clear': True, 'blacklisted': False, 'risk_level': 'LOW'},
        'property_ltv': 0.083, 'dti': 0.095,
        'expected_decision': 'APPROVE'
    },
}

print('Test cases loaded:')
for cid, data in LOAN_CASES.items():
    print(f'  {cid}: score={data["credit_score"]}, expected={data["expected_decision"]}, DTI={data["dti"]:.1%}')

In [ ]:
# ─── System Prompt Versions ────────────────────────────────────────────────────

PROMPT_V1_VAGUE = """You are a helpful loan assessment assistant. 
Review the loan application and decide if it should be approved.
Return your decision in JSON format."""

PROMPT_V2_STRUCTURED = """
## VAI TRÒ (Role)
Bạn là LoanBot v3.2 — chuyên gia thẩm định tín dụng của FinTech Corp.
Authority: phê duyệt ≤500M VNĐ | escalate L1 HITL ≤2B | escalate L2 HITL >2B.

## BỐI CẢNH (Context)
FinTech Corp: 50,000 hồ sơ/tháng, default rate 2.3%.
Khung pháp lý: TT39/2016/TT-NHNN (điều kiện tín dụng), QĐ493/2005 (phân loại nợ).

## QUY TRÌNH (Tools)
1. LUÔN kiểm tra compliance TRƯỚC các factors khác.
2. Đánh giá từng risk factor với regulation reference.
3. Weight conflicting evidence trước khi quyết định.

## RÀNG BUỘC CỨNG (Constraints)
- KHÔNG approve: blacklisted=true, credit_score<450, DTI>65%
- LUÔN cite ít nhất 1 regulation cho mỗi key_factor
- KHÔNG reveal raw credit score cho customer
- Nếu compliance FAIL → REJECT, không bao giờ APPROVE
- Uncertainty → HITL (không đoán mò)

## ĐỊNH DẠNG (Format)
Trả về ONLY valid JSON, không có markdown. Schema:
{
  "loan_id": string,
  "summary_vi": string,
  "decision": "APPROVE"|"REJECT"|"REVIEW"|"HITL",
  "risk_score": float (0.0-1.0),
  "confidence": float (0.0-1.0),
  "key_factors": [string],
  "regulation_refs": [string],
  "hitl_required": boolean,
  "hitl_level": 1|2|3|null
}
"""

print('Prompt comparison:')
v1_tokens = len(PROMPT_V1_VAGUE.split())
v2_tokens = len(PROMPT_V2_STRUCTURED.split())
print(f'  v1 (vague):      ~{v1_tokens} words')
print(f'  v2 (structured): ~{v2_tokens} words')
print(f'  Ratio: {v2_tokens/v1_tokens:.1f}x longer — but research shows ~31% accuracy improvement')

## Section 2: Chain-of-Thought Reasoning Simulation

In [ ]:
# ─── LoanBot Reasoning Engine (CoT simulation) ───────────────────────────────
# Simulates what Claude would produce with CoT instruction

@dataclass
class ReasoningStep:
    step_num: int
    name:     str
    finding:  str
    pass_fail: str  # 'PASS' | 'FAIL' | 'FLAG' | 'INFO'

@dataclass
class LoanAssessment:
    loan_id:         str
    summary_vi:      str
    decision:        str
    risk_score:      float
    confidence:      float
    key_factors:     List[str]
    regulation_refs: List[str]
    hitl_required:   bool
    hitl_level:      Optional[int] = None
    reasoning_steps: List[ReasoningStep] = field(default_factory=list)


class LoanBotCoT:
    """Simulates LoanBot reasoning with explicit Chain-of-Thought steps."""

    # Hard rule thresholds
    MIN_CREDIT_SCORE = 450
    MAX_DTI          = 0.65
    MAX_INCOME_MULT  = 10.0
    HITL_RISK_THRESH = 0.45
    APPROVE_MAX_RISK = 0.35

    def assess(self, loan_id: str, case: dict) -> LoanAssessment:
        steps   = []
        factors = []
        refs    = []

        # ── Step 1: Hard Rules Gate ────────────────────────────────────────────
        comp    = case['compliance']
        hard_fail = None

        if comp['blacklisted']:
            steps.append(ReasoningStep(1, 'Hard Rules Gate',
                f'blacklisted=True → FAIL hard constraint', 'FAIL'))
            return LoanAssessment(
                loan_id=loan_id,
                summary_vi='Từ chối: khách hàng trong danh sách cấm AML/PEP.',
                decision='REJECT', risk_score=0.97, confidence=0.99,
                key_factors=['blacklisted_violation'],
                regulation_refs=['TT39/2016 Điều 7 Khoản 3', 'Nghị định 87/2019 AML'],
                hitl_required=False, reasoning_steps=steps
            )

        if not comp['aml_clear']:
            steps.append(ReasoningStep(1, 'Hard Rules Gate',
                'aml_clear=False → FAIL compliance gate', 'FAIL'))
            return LoanAssessment(
                loan_id=loan_id,
                summary_vi='Từ chối: không vượt qua kiểm tra AML.',
                decision='REJECT', risk_score=0.95, confidence=0.98,
                key_factors=['aml_check_failed'],
                regulation_refs=['Nghị định 87/2019 AML Điều 15'],
                hitl_required=False, reasoning_steps=steps
            )

        if case['credit_score'] < self.MIN_CREDIT_SCORE:
            steps.append(ReasoningStep(1, 'Hard Rules Gate',
                f'credit_score={case["credit_score"]} < minimum {self.MIN_CREDIT_SCORE} → FAIL', 'FAIL'))
            return LoanAssessment(
                loan_id=loan_id,
                summary_vi=f'Từ chối: điểm tín dụng dưới ngưỡng tối thiểu.',
                decision='REJECT', risk_score=0.90, confidence=0.97,
                key_factors=[f'credit_score_{case["credit_score"]}_below_minimum'],
                regulation_refs=['TT39/2016 Điều 15 Khoản 1'],
                hitl_required=False, reasoning_steps=steps
            )

        steps.append(ReasoningStep(1, 'Hard Rules Gate',
            f'blacklist=False, aml=CLEAR, score={case["credit_score"]}≥{self.MIN_CREDIT_SCORE} → PASS', 'PASS'))

        # ── Step 2: Risk Factor Analysis ───────────────────────────────────────
        risk_scores = []

        # Credit score analysis
        score = case['credit_score']
        if score >= 700:
            credit_risk = 0.15; credit_label = f'credit_score_{score}_grade_A_excellent'
        elif score >= 650:
            credit_risk = 0.30; credit_label = f'credit_score_{score}_grade_B_good'
        elif score >= 580:
            credit_risk = 0.55; credit_label = f'credit_score_{score}_grade_C_moderate'
        else:
            credit_risk = 0.75; credit_label = f'credit_score_{score}_grade_D_poor'

        risk_scores.append(credit_risk)
        factors.append(credit_label)
        refs.append('TT39/2016 Điều 15')
        steps.append(ReasoningStep(2, 'Credit Analysis',
            f'score={score} → credit risk={credit_risk:.2f}', 'INFO'))

        # DTI analysis
        dti = case['dti']
        if dti > self.MAX_DTI:
            dti_risk = 0.90; dti_flag = 'FAIL'
        elif dti > 0.45:
            dti_risk = 0.65; dti_flag = 'FLAG'
        elif dti > 0.30:
            dti_risk = 0.40; dti_flag = 'INFO'
        else:
            dti_risk = 0.15; dti_flag = 'PASS'

        risk_scores.append(dti_risk)
        factors.append(f'dti_{dti:.1%}')
        refs.append('QĐ493/2005 Khoản 2')
        steps.append(ReasoningStep(2, 'DTI Analysis',
            f'DTI={dti:.1%} → dti risk={dti_risk:.2f}', dti_flag))

        # Collateral (LTV)
        ltv = case.get('property_ltv')
        if ltv is not None:
            ltv_risk = max(0.05, min(0.90, ltv * 1.5))
            factors.append(f'property_ltv_{ltv:.1%}')
            refs.append('TT39/2016 Điều 20')
            risk_scores.append(ltv_risk)
            steps.append(ReasoningStep(2, 'Collateral Analysis',
                f'LTV={ltv:.1%} → ltv risk={ltv_risk:.2f}', 'PASS' if ltv < 0.60 else 'FLAG'))
        else:
            risk_scores.append(0.50)  # no collateral = moderate risk
            factors.append('no_collateral')
            steps.append(ReasoningStep(2, 'Collateral Analysis', 'No collateral → moderate risk', 'FLAG'))

        # Income verification
        if case.get('income_verified'):
            risk_scores.append(0.10)
            factors.append('income_verified')
            refs.append('TT39/2016 Điều 15 Khoản 2')
            steps.append(ReasoningStep(2, 'Income Analysis', 'Income verified → low risk', 'PASS'))
        else:
            risk_scores.append(0.65)
            factors.append('income_unverified')
            steps.append(ReasoningStep(2, 'Income Analysis', 'Income not verified → elevated risk', 'FLAG'))

        # ── Step 3: Aggregate & Weight ─────────────────────────────────────────
        # Weighted average: credit 40%, DTI 30%, collateral 20%, income 10%
        weights = [0.40, 0.30, 0.20, 0.10]
        risk_score = sum(r * w for r, w in zip(risk_scores[:4], weights))
        risk_score = round(min(0.99, max(0.01, risk_score)), 3)

        confidence = round(1.0 - abs(risk_score - 0.5) * 0.5, 2)  # higher confidence when clear
        steps.append(ReasoningStep(3, 'Aggregate Risks',
            f'Weighted risk={risk_score:.3f} (scores={[round(r,2) for r in risk_scores[:4]]}, weights={weights})', 'INFO'))

        # ── Step 4: Decision ───────────────────────────────────────────────────
        if risk_score < self.APPROVE_MAX_RISK:
            decision = 'APPROVE'; hitl = False; hitl_level = None
        elif risk_score >= 0.70:
            decision = 'REJECT'; hitl = False; hitl_level = None
        elif risk_score >= self.HITL_RISK_THRESH:
            decision = 'HITL'; hitl = True; hitl_level = 1
        else:
            decision = 'REVIEW'; hitl = False; hitl_level = None

        steps.append(ReasoningStep(4, 'Decision',
            f'risk={risk_score:.3f} → decision={decision}', 'INFO'))

        decision_labels = {'APPROVE': 'Phê duyệt', 'REJECT': 'Từ chối',
                           'REVIEW': 'Cần xem xét thêm', 'HITL': 'Cần thẩm định thủ công'}
        summary = f'{decision_labels[decision]}: risk_score={risk_score:.2f}, credit={score}, DTI={dti:.1%}.'

        return LoanAssessment(
            loan_id=loan_id, summary_vi=summary, decision=decision,
            risk_score=risk_score, confidence=confidence,
            key_factors=factors, regulation_refs=list(set(refs)),
            hitl_required=hitl, hitl_level=hitl_level,
            reasoning_steps=steps
        )


# Run CoT assessment for all 4 customers
bot = LoanBotCoT()
assessments = {}

print('=== Chain-of-Thought Assessment ===')
for cid, case in LOAN_CASES.items():
    result = bot.assess(cid, case)
    assessments[cid] = result
    expected = case['expected_decision']
    match = '✅' if result.decision == expected else '⚠️ '
    print(f'\n{match} {cid}: {result.decision} (expected {expected})')
    print(f'   Risk: {result.risk_score:.3f} | Confidence: {result.confidence:.2f}')
    print(f'   Summary: {result.summary_vi}')
    print(f'   Reasoning steps:')
    for step in result.reasoning_steps:
        icon = {'PASS': '✅', 'FAIL': '❌', 'FLAG': '⚠️', 'INFO': 'ℹ️'}.get(step.pass_fail, '?')
        print(f'     {icon} Step {step.step_num} [{step.name}]: {step.finding}')

## Section 3: Few-Shot Exemplar Design

In [ ]:
# ─── Few-Shot Exemplars ────────────────────────────────────────────────────────

# Convert assessment to JSON string (as model would produce)
def assessment_to_json(a: LoanAssessment) -> str:
    data = {
        'loan_id':         a.loan_id,
        'summary_vi':      a.summary_vi,
        'decision':        a.decision,
        'risk_score':      a.risk_score,
        'confidence':      a.confidence,
        'key_factors':     a.key_factors,
        'regulation_refs': a.regulation_refs,
        'hitl_required':   a.hitl_required,
        'hitl_level':      a.hitl_level,
    }
    return json.dumps(data, ensure_ascii=False, indent=2)

def format_loan_input(loan_id: str, case: dict) -> str:
    comp = case['compliance']
    return (
        f"Thẩm định hồ sơ {loan_id}:\n"
        f"- Số tiền vay: {case['amount_vnd']:,} VNĐ\n"
        f"- Điểm tín dụng: {case['credit_score']} (grade {case['credit_grade']})\n"
        f"- Thu nhập tháng: {case['monthly_income']:,} VNĐ (verified={case['income_verified']})\n"
        f"- Compliance: aml_clear={comp['aml_clear']}, blacklisted={comp['blacklisted']}, risk={comp['risk_level']}\n"
        f"- DTI: {case['dti']:.1%}\n"
        f"- LTV: {case['property_ltv']:.1%}" if case.get('property_ltv') else "- Tài sản thế chấp: không có"
    )


# Build few-shot message set from our 3 standard exemplars
# (exclude FC-2024-004 — that's our test case)
FEW_SHOT_MESSAGES = [
    {'role': 'user',      'content': format_loan_input('FC-2024-001', LOAN_CASES['FC-2024-001'])},
    {'role': 'assistant', 'content': assessment_to_json(assessments['FC-2024-001'])},
    {'role': 'user',      'content': format_loan_input('FC-2024-002', LOAN_CASES['FC-2024-002'])},
    {'role': 'assistant', 'content': assessment_to_json(assessments['FC-2024-002'])},
    {'role': 'user',      'content': format_loan_input('FC-2024-003', LOAN_CASES['FC-2024-003'])},
    {'role': 'assistant', 'content': assessment_to_json(assessments['FC-2024-003'])},
]

print('=== Few-Shot Message Structure ===')
print(f'Total messages: {len(FEW_SHOT_MESSAGES)} (3 user + 3 assistant pairs)')
print(f'Decision coverage: APPROVE + REVIEW + REJECT (all 3 types represented)')
print()
for i, msg in enumerate(FEW_SHOT_MESSAGES):
    role   = msg['role']
    preview = msg['content'][:80].replace('\n', ' ') + '...'
    print(f'  [{i+1}] {role}: {preview}')

print()
print('=== Exemplar Quality Check ===')
decisions = [json.loads(m['content'])['decision']
             for m in FEW_SHOT_MESSAGES if m['role'] == 'assistant']
print(f'Decisions in exemplars: {decisions}')
has_reject  = 'REJECT'  in decisions
has_approve = 'APPROVE' in decisions
has_review  = any(d in ('REVIEW', 'HITL') for d in decisions)
print(f'Has REJECT: {has_reject}  {'✅' if has_reject else '❌ WARNING'}')
print(f'Has APPROVE: {has_approve} {'✅' if has_approve else '❌ WARNING'}')
print(f'Has REVIEW/HITL: {has_review} {'✅' if has_review else '❌ WARNING — add HITL example'}')

## Section 4: Structured Output Parsing & Validation

In [ ]:
# ─── Structured Output Parser ─────────────────────────────────────────────────

class OutputParseError(Exception):
    pass

class OutputValidationError(Exception):
    pass


def parse_structured_output(raw: str) -> dict:
    """Parse JSON from model output, handling markdown code fences."""
    cleaned = raw.strip()

    # Strip markdown code fences if present
    if cleaned.startswith('```'):
        lines = cleaned.split('\n')
        # Remove first line (```json or ```) and last line (```)
        cleaned = '\n'.join(lines[1:-1]).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise OutputParseError(f'Invalid JSON: {e}. Raw: {raw[:200]}')


def validate_loan_decision(data: dict) -> dict:
    """Validate business rules after JSON parse."""
    required = ['loan_id', 'decision', 'risk_score', 'confidence',
                'key_factors', 'regulation_refs', 'hitl_required']

    for field_name in required:
        if field_name not in data:
            raise OutputValidationError(f'Missing required field: {field_name}')

    if data['decision'] not in ('APPROVE', 'REJECT', 'REVIEW', 'HITL'):
        raise OutputValidationError(f'Invalid decision: {data["decision"]}')

    if not 0.0 <= data['risk_score'] <= 1.0:
        raise OutputValidationError(f'risk_score out of range: {data["risk_score"]}')

    if not data.get('regulation_refs'):  # EU AI Act compliance
        raise OutputValidationError('regulation_refs cannot be empty (EU AI Act Art.13)')

    # Business logic consistency check
    if data['decision'] == 'APPROVE' and data['risk_score'] > 0.60:
        raise OutputValidationError(f'Inconsistent: APPROVE with risk_score={data["risk_score"]:.2f}')

    if data['decision'] == 'REJECT' and data['risk_score'] < 0.40:
        raise OutputValidationError(f'Inconsistent: REJECT with risk_score={data["risk_score"]:.2f}')

    return data


def correction_prompt(original_raw: str, error: str) -> str:
    """Generate correction prompt for retry when output is invalid."""
    return (
        f"Your previous response was invalid: {error}\n\n"
        f"Here's what you returned (first 300 chars):\n{original_raw[:300]}\n\n"
        f"Please return ONLY valid JSON per the schema. No markdown, no explanation."
    )


# Test parsing with various outputs
print('=== Parser Test Cases ===')

test_cases = [
    # Case 1: Clean JSON
    ('{"loan_id": "FC-2024-004", "decision": "APPROVE", "risk_score": 0.26, "confidence": 0.87, "key_factors": ["ltv_low"], "regulation_refs": ["TT39 Điều 15"], "hitl_required": false, "hitl_level": null, "summary_vi": "Tốt"}',
     'Clean JSON'),

    # Case 2: With markdown fences
    ('```json\n{"loan_id": "FC-2024-004", "decision": "APPROVE", "risk_score": 0.26, "confidence": 0.87, "key_factors": ["ltv_low"], "regulation_refs": ["TT39 Điều 15"], "hitl_required": false, "hitl_level": null, "summary_vi": "Tốt"}\n```',
     'With markdown fences'),

    # Case 3: Invalid JSON
    ('{"loan_id": "FC-2024-004" INVALID',
     'Invalid JSON'),

    # Case 4: Missing regulation_refs
    ('{"loan_id": "FC-2024-004", "decision": "APPROVE", "risk_score": 0.26, "confidence": 0.87, "key_factors": ["x"], "regulation_refs": [], "hitl_required": false, "hitl_level": null, "summary_vi": "Tốt"}',
     'Empty regulation_refs'),

    # Case 5: APPROVE with high risk (inconsistent)
    ('{"loan_id": "FC-2024-004", "decision": "APPROVE", "risk_score": 0.75, "confidence": 0.60, "key_factors": ["x"], "regulation_refs": ["TT39 Điều 15"], "hitl_required": false, "hitl_level": null, "summary_vi": "Tốt"}',
     'APPROVE with risk=0.75 (inconsistent)'),
]

for raw, label in test_cases:
    print(f'\n  Test: {label}')
    try:
        data = parse_structured_output(raw)
        validated = validate_loan_decision(data)
        print(f'  ✅ Valid: decision={validated["decision"]}, risk={validated["risk_score"]}')
    except OutputParseError as e:
        print(f'  ❌ ParseError: {str(e)[:80]}')
    except OutputValidationError as e:
        print(f'  ⚠️  ValidationError: {e}')

## Section 5: Self-Critique Loop & A/B Test Simulation

In [ ]:
# ─── Self-Critique Loop ────────────────────────────────────────────────────────

def needs_critique(assessment: LoanAssessment) -> bool:
    """Only apply self-critique for borderline cases."""
    return 0.35 <= assessment.risk_score <= 0.65


def simulate_critique(assessment: LoanAssessment) -> LoanAssessment:
    """
    Simulates self-critique step.
    In production: calls LLM with critique prompt.
    Here: applies rule-based critique.
    """
    issues = []
    improved = False

    # Critique 1: Sufficient regulation refs?
    if len(assessment.regulation_refs) < 2:
        issues.append(f'Only {len(assessment.regulation_refs)} regulation ref — need ≥2')
        assessment.regulation_refs.append('QĐ493/2005 Khoản 2')  # add default
        improved = True

    # Critique 2: Decision-risk consistency?
    if assessment.decision == 'APPROVE' and assessment.risk_score > 0.55:
        issues.append(f'APPROVE with risk={assessment.risk_score:.2f} > 0.55 — inconsistent!')
        assessment.decision = 'REVIEW'  # downgrade to safer option
        improved = True

    # Critique 3: HITL missing reason?
    if assessment.hitl_required and not any('hitl_reason' in str(f) for f in assessment.key_factors):
        assessment.key_factors.append('hitl_reason: cần xác minh thêm thông tin')
        improved = True

    status = '🔧 IMPROVED' if improved else '✅ NO CHANGES'
    print(f'  Critique result: {status}')
    if issues:
        for issue in issues:
            print(f'    - Found: {issue}')

    return assessment


print('=== Self-Critique Loop ===')
for cid, assessment in assessments.items():
    if needs_critique(assessment):
        print(f'\n{cid}: risk={assessment.risk_score:.3f} → IN BORDERLINE ZONE → applying critique')
        assessments[cid] = simulate_critique(assessment)
        print(f'  After: decision={assessments[cid].decision}, risk={assessments[cid].risk_score}')
    else:
        print(f'{cid}: risk={assessment.risk_score:.3f} → CLEAR — skipping critique (save cost)')

In [ ]:
# ─── Prompt A/B Test Simulation ────────────────────────────────────────────────

import statistics

def simulate_prompt_ab_test(prompt_a_name: str, prompt_b_name: str,
                             n_cases: int = 100, seed: int = 42) -> dict:
    """
    Simulates A/B test between two prompts.
    In production: runs both prompts through LLM, LLM-as-Judge picks winner.
    Here: simulates with scoring based on prompt quality.
    """
    rng = random.Random(seed)

    # Simulate: structured prompt wins more often
    # v1 (vague): wins 37% cases (random-ish)
    # v2 (structured): wins 63% cases
    b_win_rate = 0.63 if 'STRUCTURED' in prompt_b_name.upper() or 'V2' in prompt_b_name else 0.50

    results = []
    metrics = {'regulation_score_a': [], 'regulation_score_b': [],
               'consistency_a': [], 'consistency_b': []}

    for i in range(n_cases):
        b_wins = rng.random() < b_win_rate
        results.append({'winner': prompt_b_name if b_wins else prompt_a_name, 'case': i})

        # Simulate metric scores
        metrics['regulation_score_a'].append(rng.uniform(0.1, 0.4))
        metrics['regulation_score_b'].append(rng.uniform(0.7, 1.0))
        metrics['consistency_a'].append(rng.uniform(0.5, 0.8))
        metrics['consistency_b'].append(rng.uniform(0.80, 0.99))

    b_wins_count = sum(1 for r in results if r['winner'] == prompt_b_name)
    a_wins_count = n_cases - b_wins_count

    # Approximate chi-squared test for proportion difference
    expected = n_cases * 0.5
    chi2 = (b_wins_count - expected)**2 / expected + (a_wins_count - expected)**2 / expected
    # Approximate p-value for chi2 with 1 df (normal approximation)
    z = (abs(b_wins_count - a_wins_count) / (n_cases**0.5)) * 0.5
    p_approx = max(0.001, 2 * (1 - (1 - 2.718**(-z**2/2) / (2 * 3.14159)**0.5)))

    return {
        'n_cases': n_cases,
        f'{prompt_a_name}_wins': a_wins_count,
        f'{prompt_b_name}_wins': b_wins_count,
        f'{prompt_b_name}_win_rate': round(b_wins_count / n_cases, 3),
        'chi2': round(chi2, 2),
        'p_approx': round(p_approx, 4),
        'significant_p05': p_approx < 0.05,
        'avg_regulation_refs_a': round(statistics.mean(metrics['regulation_score_a']), 3),
        'avg_regulation_refs_b': round(statistics.mean(metrics['regulation_score_b']), 3),
        'avg_consistency_a':     round(statistics.mean(metrics['consistency_a']), 3),
        'avg_consistency_b':     round(statistics.mean(metrics['consistency_b']), 3),
        'recommendation': f'PROMOTE {prompt_b_name}' if b_wins_count > a_wins_count and p_approx < 0.05 else 'KEEP current'
    }


print('=== Prompt A/B Test: v1_vague vs v2_structured ===')
result = simulate_prompt_ab_test('PROMPT_V1', 'PROMPT_V2_STRUCTURED', n_cases=500)

print(f'\nTest results (n={result["n_cases"]} cases):')
print(f'  PROMPT_V1 wins:          {result["PROMPT_V1_wins"]} ({result["PROMPT_V1_wins"]/result["n_cases"]:.1%})')
print(f'  PROMPT_V2_STRUCTURED wins: {result["PROMPT_V2_STRUCTURED_wins"]} ({result["PROMPT_V2_STRUCTURED_wins"]/result["n_cases"]:.1%})')
print(f'  Chi²: {result["chi2"]}, p≈{result["p_approx"]}, significant: {result["significant_p05"]}')
print(f'\nMetric comparison:')
print(f'  Regulation refs quality: V1={result["avg_regulation_refs_a"]:.3f} vs V2={result["avg_regulation_refs_b"]:.3f}')
print(f'  Decision consistency:    V1={result["avg_consistency_a"]:.3f} vs V2={result["avg_consistency_b"]:.3f}')
print(f'\n📋 Recommendation: {result["recommendation"]}')

## Section 6: Bài tập — Cải thiện Prompt cho Edge Case

In [ ]:
# ─── BÀI TẬP: Prompt Improvement for Edge Cases ─────────────────────────────
#
# LoanBot gặp edge case: khách hàng có credit_score=650 (borderline B)
# nhưng income=100M VNĐ/tháng (rất cao), amount=200M (DTI=1.4% — cực thấp)
# và property_ltv=0.05 (LTV chỉ 5%!)
#
# Current LoanBot với PROMPT_V2_STRUCTURED assess: REVIEW (risk=0.38)
# Expected: APPROVE — income và collateral bù đắp hoàn toàn borderline score
#
# Nhiệm vụ: Thêm rule vào PROMPT_V2_STRUCTURED để handle edge case này:
# "Nếu DTI < 5% VÀ LTV < 10%, credit_score >= 600 có thể APPROVE
#  (exceptional collateral & income override borderline score)"
#
# Bước 1: Add rule vào CONSTRAINTS section
# Bước 2: Add regulation reference (TT39/2016 Điều 12 — collateral override)
# Bước 3: Test trên edge case
# Bước 4: Verify không break existing cases

EDGE_CASE = {
    'amount_vnd': 200_000_000,
    'credit_score': 650, 'credit_grade': 'B',
    'monthly_income': 100_000_000, 'income_verified': True,
    'compliance': {'aml_clear': True, 'blacklisted': False, 'risk_level': 'LOW'},
    'property_ltv': 0.048, 'dti': 0.014,  # DTI chỉ 1.4%!
    'expected_decision': 'APPROVE'  # should be approved despite borderline score
}

print('=== Edge Case Assessment (Before Fix) ===')
result_before = bot.assess('FC-EDGE-001', EDGE_CASE)
print(f'Current: {result_before.decision} (expected APPROVE)')
print(f'Risk: {result_before.risk_score}, Confidence: {result_before.confidence}')
print(f'Issue: credit_score=650 drags risk up despite DTI=1.4% and LTV=4.8%')

print('\n=== TODO: Implement fix in LoanBotCoTFixed ===')
print('Add to Step 2 (Risk Factor Analysis):')
print('  if case["dti"] < 0.05 and (ltv or 1.0) < 0.10 and case["credit_score"] >= 600:')
print('    → Apply collateral_override: reduce overall risk_score by 0.15')
print('    → Add regulation_ref: "TT39/2016 Điều 12 — Collateral Override"')
print()
print('Implement LoanBotCoTFixed below and verify:')
print('  ✓ FC-EDGE-001: APPROVE (was REVIEW)')
print('  ✓ FC-2024-001: still APPROVE')
print('  ✓ FC-2024-003: still REJECT (blacklisted — hard rule not affected)')

In [ ]:
# ─── TODO: Implement LoanBotCoTFixed ─────────────────────────────────────────

class LoanBotCoTFixed(LoanBotCoT):
    """
    TODO: Override assess() to add collateral override rule.
    Rule: if DTI < 5% AND LTV < 10% AND credit_score >= 600
          → exceptional collateral override → reduce risk by 0.15
          → add regulation_ref: TT39/2016 Điều 12
    """

    def assess(self, loan_id: str, case: dict):
        # Call parent assess first
        result = super().assess(loan_id, case)

        # TODO: Check for exceptional collateral condition
        # dti      = case['dti']
        # ltv      = case.get('property_ltv', 1.0)
        # score    = case['credit_score']
        # if ...:  # fill in condition
        #     result.risk_score = max(0.01, result.risk_score - 0.15)
        #     result.regulation_refs.append('TT39/2016 Điều 12 — Collateral Override')
        #     result.key_factors.append('exceptional_collateral_override')
        #     # Recalculate decision based on new risk_score
        #     if result.risk_score < self.APPROVE_MAX_RISK:
        #         result.decision = 'APPROVE'

        return result

# Test your fix
fixed_bot = LoanBotCoTFixed()
print('=== Test After Fix ===')

# Edge case
result_fixed = fixed_bot.assess('FC-EDGE-001', EDGE_CASE)
check = '✅' if result_fixed.decision == 'APPROVE' else '❌'
print(f'{check} FC-EDGE-001: {result_fixed.decision} (expected APPROVE), risk={result_fixed.risk_score:.3f}')

# Regression check
for cid in ['FC-2024-001', 'FC-2024-003']:
    result_reg = fixed_bot.assess(cid, LOAN_CASES[cid])
    expected   = LOAN_CASES[cid]['expected_decision']
    check = '✅' if result_reg.decision == expected else '❌'
    print(f'{check} {cid}: {result_reg.decision} (expected {expected}) — regression check')